# TabPFN-v2-style surrogate on the BNN prior (overnight pilot)

## Description

**Question.** Is the TabPFN-v2-style surrogate (`surrogates/tabpfn.py`: one token per cell, feature and
item attention, 32 thinking rows, ~4.7M parameters) competitive with the PFNs4BO-style PFN on the BNN
prior, **at equal training steps**? Issue #1.

**Setup.** Trained like `pfn_large`: the same prior and ECDF, d = 1…6 (batch-uniform), n_train in [1, 100],
256 test points, batch 64, AdamW + warmup-cosine, bf16. One night on ulysses (2026-10-21 20:00 →
2026-10-22 07:30, the schedule fitted to the deadline). The timing probe gave about 0.39 s/step, so about
105k steps (pfn_large: 1.5M steps in ~24.5 h). The comparison uses the held-out validation sets both
runs share (seeds 10,000 + d, n ∈ {5, 20, 50}, 64 functions × 200 points): TabPFN's curve against
`pfn_large`'s stored history *at the same step*, and against `pfn_large`'s final values for scale.
Afterwards: `eval_pfn` on the new checkpoint, the same report as notebook 03.

**What to look for.**
- Does TabPFN's held-out NLL fall faster or slower per step than the PFN's? Per step it costs about 7×
  the PFN's tokens.
- Do higher d (where cell tokens and feature attention should help) gain more than low d?
- Is it stable under bf16?
- In 1-d: the same failure modes as notebook 03 (bimodal PPD far from data, missed narrow dips)?

**GO / NO-GO** (on the matched-step comparison at the end of the night, n = 20):
- GO for a longer run if TabPFN's NLL is at least as good as pfn_large's at the same step for every d ≥ 4;
- NO-GO if it is worse by more than 0.1 nats at every d;
- otherwise INCONCLUSIVE.

**Limits.**
- One seed and one size, so a NO-GO speaks against this configuration, not the architecture.
- The matched-step comparison ignores wall time: per step, TabPFN is ~2× slower here than pfn_large
  was (0.39 vs ~0.16 s/step) on the same GPU.


In [ ]:
from pfn_repro.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/tabpfn_bnn/<sweep_id>"  # set after `just snapshot runs/tabpfn_bnn/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.